In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Carregar os dados divididos anteriormente
train_df = pd.read_csv('../data/processed/train_classification.csv')
test_df = pd.read_csv('../data/processed/test_classification.csv')

X_train = train_df.drop(columns=['Addiction_Level'])
y_train = train_df['Addiction_Level']
X_test = test_df.drop(columns=['Addiction_Level'])
y_test = test_df['Addiction_Level']

# 2. Instanciar e treinar a Árvore de Decisão (Sem limites)
# Usaremos o critério 'gini' por padrão e semente 42 para reprodutibilidade
tree_clf_unconstrained = DecisionTreeClassifier(criterion='gini', random_state=42)
tree_clf_unconstrained.fit(X_train, y_train)

# 3. Prever e avaliar no conjunto de Treino e de Teste
y_pred_train = tree_clf_unconstrained.predict(X_train)
y_pred_test = tree_clf_unconstrained.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Profundidade da árvore criada: {tree_clf_unconstrained.get_depth()} níveis")
print(f"Acurácia no Treino: {acc_train * 100:.2f}%")
print(f"Acurácia no Teste: {acc_test * 100:.2f}%")

Profundidade da árvore criada: 18 níveis
Acurácia no Treino: 100.00%
Acurácia no Teste: 67.33%


In [2]:
from sklearn.model_selection import GridSearchCV

# Definir a grade de hiperparâmetros para teste
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10], # Poda: limitando a profundidade
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

# Configurar a busca exaustiva com Validação Cruzada (CV=5)
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42), 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)

# Treinar testando todas as combinações da grade
grid_search.fit(X_train, y_train)

# Extrair a melhor árvore podada
best_tree = grid_search.best_estimator_

# Novas previsões
y_pred_train_best = best_tree.predict(X_train)
y_pred_test_best = best_tree.predict(X_test)

acc_train_best = accuracy_score(y_train, y_pred_train_best)
acc_test_best = accuracy_score(y_test, y_pred_test_best)

print(f"Melhores parâmetros encontrados:\n{grid_search.best_params_}")
print(f"Nova Profundidade da árvore: {best_tree.get_depth()} níveis")
print(f"Acurácia no Treino (Podada): {acc_train_best * 100:.2f}%")
print(f"Acurácia no Teste (Podada): {acc_test_best * 100:.2f}%")

Melhores parâmetros encontrados:
{'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 2}
Nova Profundidade da árvore: 3 níveis
Acurácia no Treino (Podada): 76.33%
Acurácia no Teste (Podada): 76.33%


In [3]:
from sklearn.metrics import f1_score, roc_auc_score

# Previsão de probabilidades exigida pelo cálculo do ROC-AUC
y_prob_test_best = best_tree.predict_proba(X_test)

# Cálculo Macro para classificação multiclasse
f1 = f1_score(y_test, y_pred_test_best, average='macro')
roc_auc = roc_auc_score(y_test, y_prob_test_best, multi_class='ovr', average='macro')

print(f"F1-Score (Macro): {f1:.4f}")
print(f"ROC-AUC (Macro OVR): {roc_auc:.4f}")

F1-Score (Macro): 0.7629
ROC-AUC (Macro OVR): 0.9195
